In [ ]:
!pip install roboflow -q
!pip install ultralytics==8.1.0 -q
!pip install albumentations==1.3.1 -q
!pip install realesrgan basicsr -q
!pip install opencv-python-headless -q

import subprocess
print('=== GPU Status ===')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout)
print('All packages installed.')

import torch
if not hasattr(torch.load, '_patched'):
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_load(*args, **kwargs)
    _patched_load._patched = True
    torch.load = _patched_load

import os
os.environ['WANDB_MODE'] = 'disabled'
from ultralytics.utils import SETTINGS
SETTINGS['wandb'] = False

In [ ]:
import os, cv2, json, shutil, random, warnings, yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import albumentations as A
from ultralytics import YOLO
import torch
import numpy as np
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid
try:
    import ray.tune as _ray_tune
    if not hasattr(_ray_tune, 'is_session_enabled'):
        _ray_tune.is_session_enabled = lambda: False
except ImportError:
    pass
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

WORK_DIR = Path('/kaggle/working')
RUNS_DIR = WORK_DIR / 'runs'
RUNS_DIR.mkdir(exist_ok=True)

IMG_SIZE   = 640
BATCH_SIZE = 16  
EPOCHS     = 35

print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
from roboflow import Roboflow

RF_API_KEY = ''  
rf      = Roboflow(api_key=RF_API_KEY)
project = rf.workspace('camouflagedetection').project('camouflage-detection-girgk')
dataset = project.version(1).download('yolov8')

DS_DIR    = Path(dataset.location)
DATA_YAML = DS_DIR / 'data.yaml'

with open(DATA_YAML) as f:
    info = yaml.safe_load(f)

print(f'Dataset location : {DS_DIR}')
print(f'Classes          : {info["names"]}')
print(f'NC               : {info["nc"]}')

for split in ['train', 'valid', 'test']:
    img_dir = DS_DIR / split / 'images'
    if not img_dir.exists():
        img_dir = DS_DIR / 'images' / split
    if img_dir.exists():
        n = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
        print(f'  {split:6s}: {n} images')

In [ ]:
def find_img_lbl_dirs(ds_root, split):
    root = Path(ds_root)
    if (root / split / 'images').exists():
        return root / split / 'images', root / split / 'labels'
    if (root / 'images' / split).exists():
        return root / 'images' / split, root / 'labels' / split
    return None, None

train_img_dir, train_lbl_dir = find_img_lbl_dirs(DS_DIR, 'train')

sample_paths = random.sample(
    list(train_img_dir.glob('*.jpg')),
    min(6, len(list(train_img_dir.glob('*.jpg'))))
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, p in zip(axes.flatten(), sample_paths):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = train_lbl_dir / (p.stem + '.txt')
    vis = img.copy()
    if lbl.exists():
        for line in lbl.read_text().strip().split('\n'):
            if not line.strip(): continue
            parts = line.strip().split()
            if len(parts) < 5: continue
            cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
            x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
            cv2.rectangle(vis, (x1,y1), (x2,y2), (0,255,100), 2)
            cv2.putText(vis, 'camouflage', (x1, max(y1-5,0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,100), 1)
    ax.imshow(vis); ax.set_title(p.stem[:22], fontsize=8); ax.axis('off')

plt.suptitle('Training Samples — Bounding Box Annotations', fontsize=13)
plt.tight_layout()
plt.savefig(WORK_DIR/'samples.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved samples.png')

In [ ]:
AUG_PER_IMAGE = 1  

def get_augmentation_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.30),
        A.Perspective(scale=(0.03, 0.08), p=0.25),
        A.RandomScale(scale_limit=(-0.2, 0.4), p=0.25),
        A.RandomFog(fog_coef_lower=0.10, fog_coef_upper=0.40,
                    alpha_coef=0.08, p=0.25),
        A.RandomBrightnessContrast(brightness_limit=(-0.50, 0.15),
                                   contrast_limit=(-0.20, 0.30), p=0.50),
        A.RandomGamma(gamma_limit=(60, 130), p=0.30),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25,
                             val_shift_limit=20, p=0.30),
        A.GaussNoise(var_limit=(10.0, 50.0), mean=0, p=0.40),
        A.MotionBlur(blur_limit=(3, 9), p=0.35),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.ImageCompression(quality_lower=45, quality_upper=85, p=0.30),
    ], bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.30
    ))


def augment_split(img_dir, lbl_dir, out_img, out_lbl, n_aug):
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    transform = get_augmentation_pipeline()
    imgs = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    n_added = 0

    for img_path in tqdm(imgs, desc=f'Augmenting {img_dir.parent.name}'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        img = cv2.imread(str(img_path))
        if img is None: continue

        shutil.copy(img_path, out_img / img_path.name)
        if lbl_path.exists():
            shutil.copy(lbl_path, out_lbl / lbl_path.name)
        else:
            (out_lbl / (img_path.stem + '.txt')).touch()

        if n_aug == 0 or not lbl_path.exists():
            continue

        bboxes, classes = [], []
        for line in lbl_path.read_text().strip().split('\n'):
            if not line.strip(): continue
            p = line.strip().split()
            if len(p) < 5: continue
            classes.append(int(p[0]))
            bboxes.append([float(x) for x in p[1:5]])
        if not bboxes: continue

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        for i in range(n_aug):
            try:
                res = transform(image=rgb, bboxes=bboxes, class_labels=classes)
                if not res['bboxes']: continue 
                aug_bgr  = cv2.cvtColor(res['image'], cv2.COLOR_RGB2BGR)
                aug_stem = img_path.stem + f'_aug{i}'
                cv2.imwrite(str(out_img / f'{aug_stem}.jpg'), aug_bgr)
                lines = [f"{c} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}"
                         for c, b in zip(res['class_labels'], res['bboxes'])]
                with open(out_lbl / f'{aug_stem}.txt', 'w') as f:
                    f.write('\n'.join(lines))
                n_added += 1
            except Exception: pass

    return len(imgs), n_added


if AUG_PER_IMAGE > 0:
    AUG_DIR = WORK_DIR / 'augmented'
    val_img_dir, val_lbl_dir = find_img_lbl_dirs(DS_DIR, 'valid')

    print('Augmenting training split...')
    n_orig, n_aug = augment_split(
        train_img_dir, train_lbl_dir,
        AUG_DIR / 'train' / 'images',
        AUG_DIR / 'train' / 'labels',
        AUG_PER_IMAGE
    )
    print(f'  Originals: {n_orig}  |  Augmented: {n_aug}  |  Total: {n_orig+n_aug}')

    print('Copying validation split...')
    n_v, _ = augment_split(val_img_dir, val_lbl_dir,
                           AUG_DIR / 'valid' / 'images',
                           AUG_DIR / 'valid' / 'labels', 0)
    print(f'  Val images: {n_v}')

    FINAL_DIR = AUG_DIR
else:
    print('AUG_PER_IMAGE=0: skipping augmentation, using raw Roboflow data.')
    FINAL_DIR = DS_DIR

In [ ]:
if AUG_PER_IMAGE > 0:
    yaml_text = f'''
path: {FINAL_DIR}
train: train/images
val:   valid/images
nc: 1
names:
  0: camouflage
'''
    yaml_path = WORK_DIR / 'train.yaml'
    yaml_path.write_text(yaml_text.strip())
else:
    yaml_path = DATA_YAML

print(f'YAML: {yaml_path}')
print(open(yaml_path).read())

if AUG_PER_IMAGE > 0:
    n_t = len(list((FINAL_DIR/'train'/'images').glob('*.jpg')))
    n_v = len(list((FINAL_DIR/'valid'/'images').glob('*.jpg')))
else:
    t_dir = DS_DIR/'train'/'images' if (DS_DIR/'train'/'images').exists() else DS_DIR/'images'/'train'
    v_dir = DS_DIR/'valid'/'images' if (DS_DIR/'valid'/'images').exists() else DS_DIR/'images'/'valid'
    n_t = len(list(t_dir.glob('*.jpg')))
    n_v = len(list(v_dir.glob('*.jpg')))

print(f'Training on {n_t} images | Validating on {n_v} images')

In [ ]:
model = YOLO('yolov8m.pt') 
results = model.train(
    data         = str(yaml_path),
    imgsz        = IMG_SIZE,
    batch        = BATCH_SIZE,
    epochs       = EPOCHS,

    optimizer    = 'AdamW',
    lr0          = 0.001,    
    lrf          = 0.01,     
    cos_lr       = True,
    weight_decay = 0.0005,
    momentum     = 0.937,

    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,

    overlap_mask = True,     
    mask_ratio   = 4,        
    nbs          = 64,       

    mosaic       = 1.0,      
    mixup        = 0.10,
    copy_paste   = 0.10,    
    degrees      = 10.0,
    translate    = 0.10,
    scale        = 0.50,     
    shear        = 2.0,
    perspective  = 0.0005,
    fliplr       = 0.50,
    flipud       = 0.00,
    hsv_h        = 0.015,
    hsv_s        = 0.70,
    hsv_v        = 0.40,

    device       = '0',
    workers      = 4,

    project      = str(RUNS_DIR),
    name         = 'camodet',
    save         = True,
    save_period  = 10,
    exist_ok     = True,
    plots        = True,
    val          = True,
    seed         = SEED,
    verbose      = True,
)

print('Training complete.')
print(f'Best weights: {RUNS_DIR}/camodet/weights/best.pt')

In [ ]:
best_pt = RUNS_DIR / 'camodet' / 'weights' / 'best.pt'

metrics = YOLO(str(best_pt)).val(
    data     = str(yaml_path),
    split    = 'val',
    imgsz    = IMG_SIZE,
    batch    = 8,
    conf     = 0.25,
    iou      = 0.45,
    device   = '0',
    plots    = True,
    save_json= True,
    project  = str(RUNS_DIR),
    name     = 'camodet_eval',
    exist_ok = True,
)

print('\n' + '='*50)
print('EVALUATION RESULTS (validation set)')
print('='*50)
print(f'  Box mAP@50    : {metrics.box.map50:.4f}')
print(f'  Box mAP@50:95 : {metrics.box.map:.4f}')
print(f'  Precision     : {metrics.box.mp:.4f}')
print(f'  Recall        : {metrics.box.mr:.4f}')
print('='*50)


In [ ]:
pred_model = YOLO(str(best_pt))

val_img_dir = FINAL_DIR/'valid'/'images' if AUG_PER_IMAGE > 0 else (
    DS_DIR/'valid'/'images' if (DS_DIR/'valid'/'images').exists()
    else DS_DIR/'images'/'valid'
)
val_imgs = list(val_img_dir.glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, p in zip(axes.flatten(), val_imgs):
    img    = cv2.imread(str(p))
    result = pred_model(img, conf=0.25, iou=0.45, verbose=False)[0]
    ann    = cv2.cvtColor(
               result.plot(boxes=True, conf=True, line_width=2),
               cv2.COLOR_BGR2RGB)
    n_det  = len(result.boxes) if result.boxes else 0
    ax.imshow(ann)
    ax.set_title(f'{p.stem[:22]}  |  {n_det} target(s)', fontsize=9)
    ax.axis('off')

plt.suptitle('CamouflageNet — Validation Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR/'predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved predictions.png')

In [ ]:
import shutil

shutil.copy(best_pt, WORK_DIR / 'camouflage_best.pt')
print('camouflage_best.pt -> /kaggle/working/camouflage_best.pt')

YOLO(str(best_pt)).export(
    format='onnx', imgsz=IMG_SIZE,
    dynamic=True, simplify=True, opset=17
)
print(f'ONNX -> {best_pt.parent}/best.onnx')

shutil.make_archive(
    str(WORK_DIR / 'training_artifacts'), 'zip',
    str(RUNS_DIR / 'camodet')
)
print('training_artifacts.zip -> /kaggle/working/training_artifacts.zip')

print('\n=== DOWNLOAD THESE FILES ===')
for f in [
    WORK_DIR / 'camouflage_best.pt',
    WORK_DIR / 'training_artifacts.zip',
    WORK_DIR / 'predictions.png',
    WORK_DIR / 'samples.png',
]:
    if f.exists():
        print(f'  {f.name}: {f.stat().st_size/1024**2:.1f} MB')